# Supervised Classification: Breast Cancer Diagnosis

**Goal:** Build and evaluate supervised classifiers to predict whether a tumor is malignant or benign using the scikit-learn Wisconsin Breast Cancer dataset.

This notebook covers data inspection, preprocessing, stratified train/test split, cross-validation, comparison of Logistic Regression and Random Forest, and final test metrics: accuracy, precision, recall, F1, and ROC-AUC.

> The dataset is included with scikit-learn, so no external download is required. This is an educational demonstration, not a clinical diagnostic tool.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42

## 2. Load and inspect the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
# Original target: 0 = malignant, 1 = benign. Convert to 1 = malignant for easier interpretation.
y = (dataset.target == 0).astype(int)
target_names = {0: "benign", 1: "malignant"}

print("Dataset shape:", X.shape)
print("Missing values:", int(X.isna().sum().sum()))
print("Target counts (0=benign, 1=malignant):")
print(y.value_counts().sort_index().rename(index=target_names))

display(X.head())

## 3. Train/test split
Reserve 20% of the data for final evaluation. Stratification preserves the approximate class proportions in both sets. The test set is not used during model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Training malignant proportion:", round(y_train.mean(), 3))
print("Test malignant proportion:", round(y_test.mean(), 3))

## 4. Define models and preprocessing
Logistic Regression benefits from feature scaling. Random Forest generally does not require scaling, but both models are placed in pipelines so preprocessing is fitted only on each training fold, avoiding data leakage.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
        ))
    ])
}

## 5. Cross-validation comparison
Use 5-fold stratified cross-validation on the training data. Report the mean and standard deviation for each metric. The positive class is **malignant**.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"Model": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric.title()} mean"] = vals.mean()
        row[f"{metric.title()} std"] = vals.std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).set_index("Model")
display(cv_results.round(3))

## 6. Fit on the full training set and evaluate on the held-out test set
Select the model using cross-validation F1/ROC-AUC and the project objective. This example uses the best cross-validation F1 as a transparent default. For high-stakes applications, model choice also requires domain-specific error-cost analysis.

In [ ]:
best_model_name = cv_results["F1 mean"].idxmax()
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

test_rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_score)
    })

test_results = pd.DataFrame(test_rows).set_index("Model")
display(test_results.round(3))
print("Model selected from CV F1:", best_model_name)

## 7. Visualize test-set performance

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
test_results[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", ax=ax, ylim=(0, 1), rot=15
)
ax.set_title("Held-out test metrics by model")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    fitted_models[best_model_name], X_test, y_test,
    display_labels=["Benign", "Malignant"], cmap="Blues", ax=ax
)
ax.set_title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
ax.set_title("ROC curves (positive class: malignant)")
plt.tight_layout()
plt.show()

## 8. Detailed report for selected model

In [ ]:
selected = fitted_models[best_model_name]
selected_pred = selected.predict(X_test)
print(classification_report(
    y_test, selected_pred, labels=[0, 1],
    target_names=["Benign", "Malignant"], zero_division=0
))
print("Selected model test ROC-AUC:",
      round(roc_auc_score(y_test, selected.predict_proba(X_test)[:, 1]), 4))

## 9. Interpretation and limitations

- **Accuracy** is the fraction of all predictions that are correct.
- **Precision** for malignant cases is the fraction of predicted malignant cases that are truly malignant.
- **Recall** for malignant cases is the fraction of all malignant cases detected by the model.
- **F1** balances precision and recall.
- **ROC-AUC** measures how well the score ranks malignant cases above benign cases across thresholds.
- Cross-validation estimates training-data generalization; the held-out test set gives a final, separate evaluation.
- A single split can vary. For a more robust project, repeat the experiment with multiple random seeds or nested cross-validation.
- This dataset is small and curated. Results should not be generalized to real-world clinical practice.